In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np, lightgbm as lgb
from sklearn.metrics import average_precision_score, precision_recall_curve, precision_score, recall_score

project_dir = '/content/drive/MyDrive/ringsentinel'

device_rings = pd.read_parquet(f'{project_dir}/data/device_ring_assignments.parquet')
df_baseline = pd.read_parquet(f'{project_dir}/data/baseline_features.parquet')

print(device_rings.shape, df_baseline.shape)

Mounted at /content/drive
(118666, 3) (590540, 23)


In [2]:
device_rings = device_rings.merge(
    df_baseline[['TransactionID', 'TransactionDT', 'isFraud']],
    on='TransactionID', how='left'
)

In [7]:
df_dev_sorted = device_rings.sort_values(['device_ring_id', 'TransactionDT']).copy()
df_dev_sorted['ring_fraud_count_so_far'] = (
    df_dev_sorted.groupby('device_ring_id')['isFraud'].cumsum().shift(1).fillna(0)
)
df_dev_sorted['ring_txn_count_so_far'] = (
    df_dev_sorted.groupby('device_ring_id').cumcount()
)

prior_rate = df_dev_sorted['isFraud'].mean()
smoothing_k = 5
df_dev_sorted['ring_fraud_rate_so_far'] = (
    (df_dev_sorted['ring_fraud_count_so_far'] + smoothing_k * prior_rate) /
    (df_dev_sorted['ring_txn_count_so_far'] + smoothing_k)
)

df_dev_sorted['ring_time_since_prev'] = (
    df_dev_sorted.groupby('device_ring_id')['TransactionDT'].diff()
)
df_dev_sorted['ring_time_since_prev'] = df_dev_sorted['ring_time_since_prev'].fillna(1e9)

ring_features = df_dev_sorted[[
    'TransactionID', 'device_ring_id', 'device_ring_size',
    'ring_txn_count_so_far', 'ring_fraud_rate_so_far',
    'ring_time_since_prev'
]]

ring_features.to_parquet(f'{project_dir}/data/ring_features_leak_free.parquet')
print(ring_features.shape)
ring_features.describe()

(118666, 6)


,TransactionID,device_ring_id,device_ring_size,ring_txn_count_so_far,ring_fraud_rate_so_far,ring_time_since_prev
count,1.186660e+05,118666.000000,118666.000000,118666.000000,118666.000000,1.186660e+05
mean,3.228734e+06,28599.844235,13.371901,6.185950,0.088188,5.731547e+08
std,1.793694e+05,21663.455616,22.619398,13.614814,0.167353,4.944095e+08
min,2.987004e+06,0.000000,1.000000,0.000000,0.003454,1.000000e+00
25%,3.072938e+06,7243.000000,1.000000,0.000000,0.045332,1.168265e+05
50%,3.178699e+06,27147.500000,1.000000,0.000000,0.072531,1.000000e+09
75%,3.387323e+06,48532.750000,14.000000,5.000000,0.072531,1.000000e+09
max,3.577534e+06,67988.000000,102.000000,101.000000,15.072531,1.000000e+09


In [11]:
df_full = pd.read_parquet(f'{project_dir}/data/baseline_features.parquet')
df_full = df_full.merge(
    ring_features.drop(columns=['device_ring_id']),
    on='TransactionID',
    how='left'
)

df_full['device_ring_size'] = df_full['device_ring_size'].fillna(1)
df_full['ring_txn_count_so_far'] = df_full['ring_txn_count_so_far'].fillna(0)
df_full['ring_fraud_rate_so_far'] = df_full['ring_fraud_rate_so_far'].fillna(prior_rate)
df_full['ring_time_since_prev'] = df_full['ring_time_since_prev'].fillna(1e9)

print(df_full.shape)
print(df_full[['device_ring_size', 'ring_txn_count_so_far', 'ring_fraud_rate_so_far', 'ring_time_since_prev']].isna().sum())

(590540, 27)
device_ring_size          0
ring_txn_count_so_far     0
ring_fraud_rate_so_far    0
ring_time_since_prev      0
dtype: int64


In [13]:
cutoff = df_full['TransactionDT'].quantile(0.8)
train_b = df_full[df_full['TransactionDT'] <= cutoff].copy()
test_b = df_full[df_full['TransactionDT'] > cutoff].copy()

cat_cols = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'DeviceType']
for col in cat_cols:
    train_b[col] = train_b[col].astype('category')
    test_b[col] = pd.Categorical(test_b[col], categories=train_b[col].cat.categories)

feature_cols_b = [c for c in train_b.columns if c not in ['isFraud', 'TransactionDT', 'TransactionID']]

train_set_b = lgb.Dataset(train_b[feature_cols_b], label=train_b['isFraud'], categorical_feature=cat_cols)
test_set_b = lgb.Dataset(test_b[feature_cols_b], label=test_b['isFraud'], categorical_feature=cat_cols, reference=train_set_b)

params = {
    'objective': 'binary',
    'metric': 'average_precision',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'random_state': 42,
    'n_jobs': -1
}

model_b = lgb.train(
    params,
    train_set_b,
    num_boost_round=500,
    valid_sets=[test_set_b],
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)]
)

[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.148617 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3678
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035135 -> initscore=-3.312784
[LightGBM] [Info] Start training from score -3.312784
Training until validation scores don't improve for 30 rounds
[50]	valid_0's average_precision: 0.494318
[100]	valid_0's average_precision: 0.51751
[150]	valid_0's average_precision: 0.52888
[200]	valid_0's average_precision: 0.536633
[250]	valid_0's average_precision: 0.541012
[300]	valid_0's average_precision: 0.545534
[350]	valid_0's average_precision: 0.54772
[400]	valid_0's average_precision: 0.552812
[450]	valid_0's average_precision: 0.555068
[500]	valid_0's average_precision: 0.557984
Did not

In [14]:
preds_b = model_b.predict(test_b[feature_cols_b])
pr_auc_b = average_precision_score(test_b['isFraud'], preds_b)

print(f"Model A (baseline)        PR-AUC: 0.5166")
print(f"Model B (+ ring features) PR-AUC: {pr_auc_b:.4f}")
print(f"Improvement: {pr_auc_b - 0.5166:+.4f}")

for t in [0.3, 0.5, 0.7, 0.9]:
    pred_labels = (preds_b >= t).astype(int)
    p = precision_score(test_b['isFraud'], pred_labels)
    r = recall_score(test_b['isFraud'], pred_labels)
    print(f"threshold={t}: precision={p:.3f}, recall={r:.3f}")

importance = pd.Series(model_b.feature_importance(importance_type='gain'), index=feature_cols_b)
print(importance.sort_values(ascending=False).head(10))

Model A (baseline)        PR-AUC: 0.5166
Model B (+ ring features) PR-AUC: 0.5581
Improvement: +0.0415
threshold=0.3: precision=0.705, recall=0.425
threshold=0.5: precision=0.830, recall=0.363
threshold=0.7: precision=0.883, recall=0.297
threshold=0.9: precision=0.937, recall=0.201
ring_fraud_rate_so_far    141786.373672
C1                        139407.316536
C14                        58775.567025
card1                      48524.250918
D1                         48484.349916
C13                        48016.897569
card2                      47269.940993
addr1                      44181.520875
TransactionAmt             43417.479964
R_emaildomain              43212.814754
dtype: float64


In [15]:
model_b.save_model(f'{project_dir}/reports/model_b_graph_augmented.txt')
np.save(f'{project_dir}/reports/model_b_test_preds.npy', preds_b)

In [16]:
model_b.save_model(f'{project_dir}/reports/model_b_graph_augmented.txt')
np.save(f'{project_dir}/reports/model_b_test_preds.npy', preds_b)
np.save(f'{project_dir}/reports/model_b_test_labels.npy', test_b['isFraud'].values)

import json
ablation = {
    'model_a_pr_auc': 0.5166,
    'model_b_pr_auc': float(pr_auc_b),
    'improvement': float(pr_auc_b - 0.5166),
    'top_feature': 'ring_fraud_rate_so_far',
}
with open(f'{project_dir}/reports/ablation_summary.json', 'w') as f:
    json.dump(ablation, f, indent=2)
print("saved.")

saved.
